# RNN from Scratch Using Basic Python

**Classroom task:** classify a sequence as **increasing** or **decreasing**.

This notebook uses only Python's built-in data structures and the standard `math` and `random` modules. It does not use TensorFlow, PyTorch, NumPy, pandas, or matplotlib.

### Learning outcomes

Students will learn how to:

1. update an RNN hidden state one time step at a time;
2. convert the final hidden state into a prediction;
3. calculate binary cross-entropy loss;
4. train the RNN using backpropagation through time (BPTT); and
5. test whether the trained model has learned sequence order.


## 1. Why an RNN?

Compare the sequences:

- `1, 2, 3, 4, 5` → increasing
- `5, 4, 3, 2, 1` → decreasing

They contain the same values, but their order is different. An RNN processes each value while carrying a hidden state from the previous time step.

For one input and one hidden neuron:

$$a_t=W_{xh}x_t+W_{hh}h_{t-1}+b_h$$

$$h_t=\tanh(a_t)$$

After the last time step:

$$z=W_{hy}h_T+b_y, \qquad \hat y=\sigma(z)$$

Here, $\hat y$ is the predicted probability that the sequence is increasing.


In [1]:
# Only modules included with Python are used
import math
import random

random.seed(7)
print('Ready: basic Python only')


Ready: basic Python only


## 2. First perform one forward pass manually

We begin with fixed weights so every calculation can be inspected. The same weights are reused at all time steps.


In [2]:
sequence = [0.1, 0.3, 0.5]

W_xh = 1.0    # input-to-hidden weight
W_hh = 0.5    # hidden-to-hidden weight
b_h = 0.0     # hidden bias
h = 0.0       # initial hidden state

print(f'Initial hidden state: h0 = {h:.4f}\n')

for t, x_t in enumerate(sequence, start=1):
    print(t,x_t)
    a_t = W_xh * x_t + W_hh * h + b_h
    h_new = math.tanh(a_t)
    print(f't={t}: a={W_xh:.1f}({x_t:.1f}) + {W_hh:.1f}({h:.4f}) + {b_h:.1f}'
          f' = {a_t:.4f}; h=tanh(a)={h_new:.4f}')
    h = h_new


Initial hidden state: h0 = 0.0000

1 0.1
t=1: a=1.0(0.1) + 0.5(0.0000) + 0.0 = 0.1000; h=tanh(a)=0.0997
2 0.3
t=2: a=1.0(0.3) + 0.5(0.0997) + 0.0 = 0.3498; h=tanh(a)=0.3362
3 0.5
t=3: a=1.0(0.5) + 0.5(0.3362) + 0.0 = 0.6681; h=tanh(a)=0.5837


## 3. Supporting functions

The sigmoid function converts the output score into a probability between 0 and 1. Values are limited before applying the exponential to avoid numerical overflow.


In [3]:
def sigmoid(value):
    value = max(-60.0, min(60.0, value))
    return 1.0 / (1.0 + math.exp(-value))


def binary_cross_entropy(target, probability):
    probability = max(1e-12, min(1.0 - 1e-12, probability))
    return -(target * math.log(probability)
             + (1 - target) * math.log(1 - probability))


print('sigmoid(0) =', sigmoid(0.0))
print('loss when target=1 and prediction=0.8 =',
      round(binary_cross_entropy(1, 0.8), 4))


sigmoid(0) = 0.5
loss when target=1 and prediction=0.8 = 0.2231


## 4. Create the dataset

Each sequence has six time steps. A positive step produces an increasing sequence; a negative step produces a decreasing sequence. Small random variations prevent the network from merely memorizing a few examples.

- target `1`: increasing
- target `0`: decreasing

All values are kept near $[-1,1]$, where `tanh` has useful gradients.


In [4]:
def create_dataset(number_of_pairs=250, time_steps=6):
    dataset = []

    for _ in range(number_of_pairs):
        start = random.uniform(-0.5, 0.5)
        step = random.uniform(0.08, 0.18)

        increasing = []
        decreasing = []

        for t in range(time_steps):
            noise = random.uniform(-0.025, 0.025)
            increasing.append(start + step * t + noise)
            decreasing.append(start - step * t + noise)

        dataset.append((increasing, 1))
        dataset.append((decreasing, 0))

    random.shuffle(dataset)
    return dataset


dataset = create_dataset()
split = int(0.8 * len(dataset))
training_data = dataset[:split]
test_data = dataset[split:]

print('Training samples:', len(training_data))
print('Test samples:    ', len(test_data))
print('\nFirst three samples:')
for values, target in training_data[:3]:
    print([round(value, 3) for value in values], 'target =', target)


Training samples: 400
Test samples:     100

First three samples:
[-0.235, -0.447, -0.623, -0.768, -0.936, -1.149] target = 0
[-0.354, -0.41, -0.534, -0.62, -0.704, -0.772] target = 0
[-0.032, 0.065, 0.187, 0.276, 0.394, 0.5] target = 1


## 5. Define the RNN parameters

This small network has only five trainable parameters:

| Parameter | Purpose |
|---|---|
| $W_{xh}$ | current input contribution |
| $W_{hh}$ | previous hidden-state contribution |
| $b_h$ | hidden bias |
| $W_{hy}$ | hidden-to-output weight |
| $b_y$ | output bias |


In [5]:
parameters = {
    'W_xh': random.uniform(-0.5, 0.5),
    'W_hh': random.uniform(-0.5, 0.5),
    'b_h': 0.0,
    'W_hy': random.uniform(-0.5, 0.5),
    'b_y': 0.0
}

for name, value in parameters.items():
    print(f'{name:4s} = {value:+.4f}')


W_xh = -0.3848
W_hh = +0.4127
b_h  = +0.0000
W_hy = +0.2341
b_y  = +0.0000


## 6. RNN forward pass

The function stores every hidden state because BPTT needs them later. Only the final hidden state is sent to the output neuron for this many-to-one classification task.


In [6]:
def forward(sequence, parameters):
    hidden_states = [0.0]  # h0

    for x_t in sequence:
        previous_h = hidden_states[-1]
        a_t = (parameters['W_xh'] * x_t
               + parameters['W_hh'] * previous_h
               + parameters['b_h'])
        hidden_states.append(math.tanh(a_t))

    final_h = hidden_states[-1]
    output_score = parameters['W_hy'] * final_h + parameters['b_y']
    probability = sigmoid(output_score)
    return probability, hidden_states


sample_sequence, sample_target = training_data[0]
sample_probability, sample_states = forward(sample_sequence, parameters)

print('Sequence:     ', [round(x, 3) for x in sample_sequence])
print('Hidden states:', [round(h, 4) for h in sample_states])
print('Target:       ', sample_target)
print('Prediction:   ', round(sample_probability, 4))


Sequence:      [-0.235, -0.447, -0.623, -0.768, -0.936, -1.149]
Hidden states: [0.0, 0.0902, 0.2061, 0.3139, 0.4011, 0.482, 0.5656]
Target:        0
Prediction:    0.533


## 7. Backpropagation through time

For sigmoid output with binary cross-entropy loss:

$$\frac{\partial L}{\partial z}=\hat y-y$$

The output error first reaches the final hidden state. We then move backward through the stored hidden states. Since

$$\frac{d}{da}\tanh(a)=1-\tanh^2(a),$$

the error at each time step is multiplied by $1-h_t^2$ and by the recurrent weight $W_{hh}$ before reaching the preceding time step.


In [7]:
def gradients_for_one_sequence(sequence, target, parameters):
    probability, hidden_states = forward(sequence, parameters)

    gradients = {
        'W_xh': 0.0,
        'W_hh': 0.0,
        'b_h': 0.0,
        'W_hy': 0.0,
        'b_y': 0.0
    }

    # Output-layer gradients
    d_output_score = probability - target
    gradients['W_hy'] = d_output_score * hidden_states[-1]
    gradients['b_y'] = d_output_score

    # Error entering the final hidden state
    d_h = d_output_score * parameters['W_hy']

    # Move backward from the final time step to the first
    for t in range(len(sequence) - 1, -1, -1):
        h_t = hidden_states[t + 1]
        h_previous = hidden_states[t]

        d_a = d_h * (1.0 - h_t * h_t)
        gradients['W_xh'] += d_a * sequence[t]
        gradients['W_hh'] += d_a * h_previous
        gradients['b_h'] += d_a

        d_h = d_a * parameters['W_hh']

    loss = binary_cross_entropy(target, probability)
    return loss, gradients


loss, sample_gradients = gradients_for_one_sequence(
    sample_sequence, sample_target, parameters)

print('Loss:', round(loss, 4))
for name, value in sample_gradients.items():
    print(f'dL/d{name:4s} = {value:+.5f}')


Loss: 0.7615
dL/dW_xh = -0.13270
dL/dW_hh = +0.05544
dL/db_h  = +0.12643
dL/dW_hy = +0.30150
dL/db_y  = +0.53305


## 8. Train the RNN

For every sample:

1. perform the forward pass;
2. calculate the loss;
3. propagate the error backward through time;
4. clip very large gradients; and
5. update each parameter using gradient descent.

$$\theta \leftarrow \theta-\eta\frac{\partial L}{\partial\theta}$$


In [8]:
def clip(value, limit=5.0):
    return max(-limit, min(limit, value))


def train(training_data, parameters, epochs=35, learning_rate=0.03):
    loss_history = []

    for epoch in range(1, epochs + 1):
        random.shuffle(training_data)
        total_loss = 0.0

        for sequence, target in training_data:
            loss, gradients = gradients_for_one_sequence(
                sequence, target, parameters)
            total_loss += loss

            for name in parameters:
                parameters[name] -= learning_rate * clip(gradients[name])

        average_loss = total_loss / len(training_data)
        loss_history.append(average_loss)

        if epoch == 1 or epoch % 5 == 0:
            print(f'Epoch {epoch:2d} | average loss = {average_loss:.4f}')

    return loss_history


loss_history = train(training_data, parameters)


Epoch  1 | average loss = 0.5077
Epoch  5 | average loss = 0.0538
Epoch 10 | average loss = 0.0284
Epoch 15 | average loss = 0.0199
Epoch 20 | average loss = 0.0155
Epoch 25 | average loss = 0.0132
Epoch 30 | average loss = 0.0114
Epoch 35 | average loss = 0.0104


## 9. Inspect the learning history and trained parameters

The average loss should decrease. The text display below avoids any plotting library.


In [9]:
print('Loss summary')
for epoch in range(0, len(loss_history), 5):
    loss = loss_history[epoch]
    bar = '#' * max(1, int(loss * 30))
    print(f'Epoch {epoch + 1:2d}: {loss:.4f} {bar}')

print('\nTrained parameters')
for name, value in parameters.items():
    print(f'{name:4s} = {value:+.4f}')


Loss summary
Epoch  1: 0.5077 ###############
Epoch  6: 0.0451 #
Epoch 11: 0.0255 #
Epoch 16: 0.0181 #
Epoch 21: 0.0150 #
Epoch 26: 0.0130 #
Epoch 31: 0.0114 #

Trained parameters
W_xh = -4.6318
W_hh = -0.2879
b_h  = -0.2672
W_hy = -6.7220
b_y  = +0.0521


## 10. Evaluate the model

A probability of at least `0.5` is classified as increasing. Otherwise, the sequence is classified as decreasing.


In [10]:
def predict(sequence, parameters):
    probability, hidden_states = forward(sequence, parameters)
    predicted_class = 1 if probability >= 0.5 else 0
    return predicted_class, probability, hidden_states


def accuracy(dataset, parameters):
    correct = 0
    for sequence, target in dataset:
        prediction, _, _ = predict(sequence, parameters)
        if prediction == target:
            correct += 1
    return correct / len(dataset)


print(f'Training accuracy: {accuracy(training_data, parameters):.1%}')
print(f'Test accuracy:     {accuracy(test_data, parameters):.1%}')


Training accuracy: 99.8%
Test accuracy:     98.0%


In [11]:
print('Example test predictions\n')

for sequence, target in test_data[:10]:
    prediction, probability, _ = predict(sequence, parameters)
    predicted_name = 'increasing' if prediction == 1 else 'decreasing'
    actual_name = 'increasing' if target == 1 else 'decreasing'

    print([round(x, 2) for x in sequence])
    print(f'P(increasing)={probability:.3f} | '
          f'predicted={predicted_name:10s} | actual={actual_name}\n')


Example test predictions

[0.03, -0.08, -0.19, -0.32, -0.46, -0.54]
P(increasing)=0.002 | predicted=decreasing | actual=decreasing

[-0.03, -0.15, -0.28, -0.37, -0.49, -0.58]
P(increasing)=0.002 | predicted=decreasing | actual=decreasing

[-0.21, -0.32, -0.44, -0.54, -0.71, -0.81]
P(increasing)=0.001 | predicted=decreasing | actual=decreasing

[0.02, -0.15, -0.25, -0.4, -0.53, -0.68]
P(increasing)=0.001 | predicted=decreasing | actual=decreasing

[-0.35, -0.41, -0.53, -0.64, -0.7, -0.79]
P(increasing)=0.001 | predicted=decreasing | actual=decreasing

[0.21, 0.38, 0.58, 0.7, 0.9, 1.06]
P(increasing)=0.999 | predicted=increasing | actual=increasing

[0.15, -0.04, -0.2, -0.41, -0.54, -0.73]
P(increasing)=0.001 | predicted=decreasing | actual=decreasing

[0.45, 0.57, 0.72, 0.87, 1.06, 1.19]
P(increasing)=0.999 | predicted=increasing | actual=increasing

[0.47, 0.38, 0.28, 0.19, 0.06, 0.01]
P(increasing)=0.802 | predicted=increasing | actual=decreasing

[-0.26, -0.11, 0.06, 0.24, 0.43, 0.6]

## 11. Why sequence order matters

We now send the same values in their original and reversed orders. The model should change its classification.


In [12]:
def explain_prediction(sequence):
    prediction, probability, hidden_states = predict(sequence, parameters)
    name = 'increasing' if prediction == 1 else 'decreasing'
    print('Sequence:     ', [round(x, 2) for x in sequence])
    print('Hidden states:', [round(h, 3) for h in hidden_states])
    print(f'P(increasing): {probability:.3f}')
    print('Prediction:   ', name)


custom_sequence = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]

print('ORIGINAL ORDER')
explain_prediction(custom_sequence)

print('\nREVERSED ORDER')
explain_prediction(list(reversed(custom_sequence)))


ORIGINAL ORDER
Sequence:      [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
Hidden states: [0.0, 0.919, 0.375, -0.359, -0.797, -0.955, -0.992]
P(increasing): 0.999
Prediction:    increasing

REVERSED ORDER
Sequence:      [0.6, 0.4, 0.2, 0.0, -0.2, -0.4]
Hidden states: [0.0, -0.995, -0.95, -0.726, -0.058, 0.589, 0.889]
P(increasing): 0.003
Prediction:    decreasing


## 12. Try your own sequence

Edit the six values and rerun this cell.


In [13]:
my_sequence = [50, 55, 65, 70, 90, 100]

if len(my_sequence) != 6:
    raise ValueError('Enter exactly six values.')

explain_prediction(my_sequence)


Sequence:      [50, 55, 65, 70, 90, 100]
Hidden states: [0.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
P(increasing): 0.999
Prediction:    increasing


## 13. Classroom discussion and exercises

1. Why is the same $W_{xh}$ used at every time step?
2. What information is contained in `hidden_states[0]`?
3. Why must BPTT move through the time steps in reverse order?
4. Set `W_hh` to zero after training. How does the prediction change?
5. Increase the sequence length from 6 to 20. Does a single hidden neuron remain sufficient?
6. Increase the noise in `create_dataset`. How does the accuracy change?
7. Modify the dataset to classify rising and falling motor-temperature measurements.

### Final takeaway

> An RNN repeatedly combines the current input with its previous hidden state. Training adjusts the shared weights so that the hidden state retains information useful for the final prediction.

This one-neuron example is deliberately small. Practical RNNs use vectors and matrices, but the underlying sequence of operations is the same.
